# Adhoc live run — build a real PyG graph on the cluster

This notebook does **not** run pytest. `tests/` (smoke + e2e) runs `build_graph` in an
in-process `local[*]` SparkSession over tiny fixtures — hermetic, fast, and blind to
everything that only exists on a real cluster: `spark-defaults.conf`, `--py-files` /
venv shipping, executor↔GPU resource negotiation, and a work dir the executors can
actually reach.

Here we run the **actual codebase** through `bin/submit_spark_job.sh` against real data
and produce a **real `HeteroData` graph on local disk**, with its six metadata JSONs and
its node index, then read those artifacts back. Failures are reported inline, not as
assertions — this is an experiment harness, not a test suite.

Shape of a session:

1. **Preflight** — is the master up, do the workers advertise a GPU, is the venv archive built.
2. **Seed** — one `--mode full` (or `enrichment_only`) run that writes the enriched Parquet
   everything after it reuses.
3. **Experiments** — N × `--mode pyg_only` submissions, each with its own `--pyg_filename`
   and `--pyg_config`. This is the cheap loop (~5-10 min each); enrichment is not repeated.
4. **Compare** — load each `.pt` + its six metadata JSONs and diff the resulting graphs.

Every submission streams its output live into the notebook, is bounded by a timeout, and
is killed **with its process group** if it hangs — an orphaned driver holds every core on a
standalone cluster and starves every later submission.

## Configuration is entirely by environment variable

Nothing about any particular deployment is written down here — no master address, no
bucket, no account id, no data path. Export what applies to you before launching Jupyter,
and the notebook reads it:

| Variable | Required | What it is |
|---|---|---|
| `SPARK_MASTER_URL` | yes | e.g. `spark://<host>:7077`, or `local[*]` to run entirely on this machine |
| `SPARK_HOME` | usually | e.g. `/opt/spark` — a Jupyter kernel's PATH rarely has `spark-submit` |
| `PYG_SOURCE_PATHS` | for the seed run | comma-separated source directories/prefixes |
| `PYG_WORK_DIR` | no | where artifacts land; defaults to `./runs` next to this notebook |
| `SPARK_DRIVER_HOST` | multi-NIC hosts | the driver's address on the *same* network as the master |
| `SPARK_MASTER_WEB_UI` | no | defaults to `http://<master host>:8080` |
| `PYG_TIME_PERIOD` | no | `YYYY-MM` label; defaults to the current UTC month |
| `PYG_SOURCE_FORMAT` | no | `ntriples` \| `turtle_parquet` |
| `PYG_TURTLE_COLUMN` | no | forces one Turtle column name; unset, each source is resolved on its own |
| `PYG_SECTOR_BUCKET` | for the seed run | bucket holding the index-constituents CSV; see below |
| `PYG_SECTOR_KEY` | for the seed run | key for that CSV |
| `PYG_S3_ARCHIVE_BUCKET` | no | optional bucket to *additionally* mirror final artifacts to |
| `RAPIDS_JAR`, `AWS_*` | as needed | passed through to the launcher / boto3 |

**Sources that disagree on the column name.** `--source_paths` takes many prefixes in one
run, and scrapers do not always agree on what the Turtle blob column is called. Left
unset, the job resolves the name per source, so a single submission can span all of them
and produce **one** graph - which matters, because sources split across two runs land in
two `.pt` files with no edges between them.

**The constituents CSV is not optional in practice.** `PYG_SECTOR_BUCKET` /
`PYG_SECTOR_KEY` point at the index-constituents CSV, and three cross-source links
read it: the quote-to-company bridge (keyed on the company id the CSV carries, not
the ticker), the sector classification, and the same-sub-industry peer edges. Both
default to empty and **nothing fails** when they are unset - the run simply builds a
graph missing those links, with nothing logged as wrong. Only the seed run reads
them; `pyg_only` never reaches that phase.

**SEC source paths must name the filings feed.** A path under the SEC source
partition that names any other feed - or no feed at all - raises at argument-parsing
time, before Spark starts. That partition holds several feeds and only one carries
RDF; the others are crawler telemetry or raw XML that no step here can read. This is
the one way a seed submission dies instantly rather than an hour in.

**Work dir and cluster shape.** `PYG_WORK_DIR` must be reachable by every node that runs
an executor. On a multi-node cluster that means a shared mount (NFS) or an `s3a://` URI —
a plain local path resolves to a *different* disk on each machine and the commit protocol
fails. A local path is correct when the master is `local[*]`, or when every executor runs
on this host, which is the setup this notebook defaults to.

## 0. Config

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG - edit these
# ---------------------------------------------------------------------------
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import time
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

# Everything below comes from the environment. Nothing about a particular deployment --
# an address, a bucket, an account id, a data path -- is written down in this notebook.
REPO_ROOT = Path(os.environ.get(
    "PYG_REPO_ROOT", Path.home() / "application" / "pyg-knowledge-graph-builder"
))
LAUNCHER = REPO_ROOT / "bin" / "submit_spark_job.sh"
VENV_PYTHON = REPO_ROOT / ".venv" / "bin" / "python"   # has torch + torch_geometric
VENV_ARCHIVE = REPO_ROOT / "dist" / "pyspark_venv.tar.gz"

# The launcher prefers $SPARK_HOME/bin/spark-submit over PATH, which matters here: a
# Jupyter kernel inherits none of a login shell's profile, so Spark is usually not on its
# PATH at all and the failure ("spark-submit: command not found") looks nothing like a
# Spark problem.
SPARK_HOME = os.environ.get("SPARK_HOME", "")

# Required. "spark://<host>:7077" for the standalone cluster, or "local[*]" to run the
# whole pipeline in one process on this machine.
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER_URL", "")

# Optional, and only matters in client mode on a host with more than one network: the
# driver advertises an address it guesses from the first non-loopback interface, and that
# guess can land on the wrong one. Executors elsewhere then can't dial back, time out
# after 120s and are relaunched forever, while the co-located executor quietly runs the
# whole job -- the cluster looks healthy and every task lands on one node. Set this to the
# driver's address on the same network as the master.
SPARK_DRIVER_HOST = os.environ.get("SPARK_DRIVER_HOST", "")

# Master web UI, used only by preflight. Derived from the master URL when unset.
_master_host = SPARK_MASTER_URL.replace("spark://", "").partition(":")[0]
MASTER_WEB_UI = os.environ.get(
    "SPARK_MASTER_WEB_UI", f"http://{_master_host}:8080" if _master_host else ""
)

# Where the job writes: enriched Parquet, the .pt, its metadata, the node index. Defaults
# to ./runs beside this notebook. See the header note -- on a multi-node cluster this must
# be a shared mount or an s3a:// URI, because a plain path is a different disk per node.
LOCAL_WORK_DIR = os.environ.get("PYG_WORK_DIR", str(Path.cwd() / "runs"))

# Optional: mirror the FINAL artifacts (.pt + metadata + manifest) to an object store in
# addition to writing them under LOCAL_WORK_DIR.
S3_ARCHIVE_BUCKET = os.environ.get("PYG_S3_ARCHIVE_BUCKET", "")

# Source data for the seed run. Comma-separated; each entry a directory/prefix, not a
# file. Only needed for --mode full / enrichment_only, and readable by every node.
SOURCE_PATHS = os.environ.get("PYG_SOURCE_PATHS", "")
SOURCE_FORMAT = os.environ.get("PYG_SOURCE_FORMAT", "turtle_parquet")  # or ntriples
# Empty means the job resolves the Turtle column per source (build_graph's
# TURTLE_COLUMN_CANDIDATES), so one run can span sources whose scrapers disagree on the
# name. Set it only to force a single column across every source.
TURTLE_COLUMN = os.environ.get("PYG_TURTLE_COLUMN", "")
PARQUET_PARTITIONS = int(os.environ.get("PYG_PARQUET_PARTITIONS", "200"))

# The index-constituents CSV, read once on the driver during the seed run. Three
# cross-source links depend on it and all three degrade SILENTLY when it is missing:
# the company bridge keys on the company id this CSV carries, the peer edges come
# only from its sub-industry column, and sector classification falls back to a small
# built-in list. There is deliberately no built-in fallback for the id map -- guessing
# a regulator's primary key would merge two unrelated companies into one node.
SECTOR_DEFINITIONS_BUCKET = os.environ.get("PYG_SECTOR_BUCKET", "")
SECTOR_DEFINITIONS_KEY = os.environ.get("PYG_SECTOR_KEY", "")

# Period label -> year=/month= partition under LOCAL_WORK_DIR. Every experiment below
# reads and writes under this one period.
TIME_PERIOD = os.environ.get("PYG_TIME_PERIOD", datetime.now(timezone.utc).strftime("%Y-%m"))

# A hung submission must fail, not wait: an unschedulable GPU request never errors.
SEED_TIMEOUT_S = int(os.environ.get("PYG_SEED_TIMEOUT", "5400"))
EXPERIMENT_TIMEOUT_S = int(os.environ.get("PYG_EXPERIMENT_TIMEOUT", "1800"))

# Sizing profiles, one per leg, sourced by submit() for that submission only.
#
# The two legs pull in opposite directions and CANNOT share a profile. Enrichment is
# executor-heavy: hundreds of millions of triples joined out on the cluster, nothing
# large on the driver. Assembly is the reverse -- the executors only sort and join, and
# the HeteroData is built in the driver's own memory. Running assembly under the
# enrichment profile is what took the host down on 2026-08-25: the executor side had
# been promised 94 GiB of a 121 GiB box, the driver asked for the ~53 GiB of tensors it
# needed anyway, and the kernel killed the executor.
#
# Set either to "" to inherit the ambient environment instead.
SEED_PROFILE = os.environ.get(
    "PYG_SEED_PROFILE", str(REPO_ROOT / "bin" / "profiles" / "large-run.env"))
ASSEMBLY_PROFILE = os.environ.get(
    "PYG_ASSEMBLY_PROFILE", str(REPO_ROOT / "bin" / "profiles" / "pyg-assembly.env"))

# RAPIDS logs a per-operator GPU/CPU decision when this is ALL. That log is the only place
# the REASON for a fallback appears, and it is enormous -- 306 MB / 1.9M lines on a
# 6.5-hour run. It is no longer the only evidence of WHERE a run executed: section 8
# counts the Gpu* operators in the Spark event log, which answers that with explain off.
RAPIDS_EXPLAIN = os.environ.get("RAPIDS_EXPLAIN", "ALL")

# How much of each submission's output to echo live. Full output is always kept in the
# returned record regardless.
STREAM_TAIL_LINES = int(os.environ.get("PYG_STREAM_TAIL_LINES", "0"))  # 0 = stream everything; N = print only lines matching
                             # STREAM_FILTER plus the last N lines on failure
# Matched against the LOG FORMAT, not one logger's name. build_graph.py formats every
# line as "[%(levelname)s] %(name)s - ...", so "] build_graph" and "] spark_jobs." between
# them catch everything this repo logs, at every level. Naming "[INFO] build_graph" alone
# dropped the whole enrichment phase and all of PyG construction from the captured log --
# 90 minutes of a 6.5-hour run with no record of which edge type the job died on. The
# seven metadata_writer lines that did survive got through on "Saved", not on their logger
# name, which is what made a filter look like a logging defect.
STREAM_FILTER = ("] build_graph", "] spark_jobs.", "ERROR", "Exception",
                 "will run on GPU", "cannot run on GPU", "Saved", "Wrote")

# ---------------------------------------------------------------------------
# REDACTION - this notebook is meant to be committed, and a committed notebook
# carries its OUTPUTS. Everything printed below goes through redact(), so a cell
# output shows structure ("s3a://<bucket>/<prefix>") instead of identity. Spark's
# own log lines are redacted the same way as they stream past.
#
# Set PYG_REDACT=0 for a private debugging session; then clear outputs before
# committing (jupyter nbconvert --clear-output --inplace <notebook>).
# ---------------------------------------------------------------------------
REDACT_OUTPUT = os.environ.get("PYG_REDACT", "1").strip().lower() not in {"0", "false", "no"}

_REDACTIONS = [
    (re.compile(r"(?i)\b(?:AKIA|ASIA)[A-Z0-9]{8,}"), "<access-key>"),
    (re.compile(r"arn:aws[^\s\"',)]*"), "<arn>"),
    (re.compile(r"\b\d{12}\b"), "<account-id>"),          # AWS account id
    (re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b"), "<host>"),
    # Swallows the KEY as well as the bucket. A prefix names things a bucket does
    # not -- the upstream feed, the vendor, the pipeline that wrote it -- so masking
    # only the host still publishes what this run reads.
    (re.compile(r"(s3a?://)[^\s,)\"']+"), r"\1<bucket>/<prefix>"),
    # A bucket named as a bare CLI argument. submit() echoes the whole command,
    # and --s3_archive_bucket carries the name with no scheme for the rule above
    # to key on, so without this it prints in the clear.
    (re.compile(r"(--[a-z0-9_]*bucket[=\s])\S+"), r"\1<bucket>"),
    # Same for a key passed as a bare argument: --*_key carries the prefix with no
    # scheme for the s3 rule above to key on.
    (re.compile(r"(--[a-z0-9_]*key[=\s])\S+"), r"\1<key>"),
    (re.compile(r"(spark://)[^:/\s,)\"']+"), r"\1<host>"),
    (re.compile(r"/(?:home|Users)/[^/\s,:)\"']+"), "/home/<user>"),
]


def redact(text):
    """Mask deployment identity in anything this notebook prints."""
    text = str(text)
    if not REDACT_OUTPUT:
        return text
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def show(*parts):
    print(redact(" ".join(str(p) for p in parts)))


show(f"redaction   {'ON' if REDACT_OUTPUT else 'OFF - clear outputs before committing'}")
show(f"repo         {REPO_ROOT}")
show(f"master       {SPARK_MASTER_URL or '(unset - export SPARK_MASTER_URL)'}")
show(f"driver host  {SPARK_DRIVER_HOST or '(unset - Spark will guess)'}")
show(f"work dir     {LOCAL_WORK_DIR}")
show(f"sources      {SOURCE_PATHS or '(unset - needed only for the seed run)'}")
show(f"period       {TIME_PERIOD}")
show(f"seed profile {Path(SEED_PROFILE).name if SEED_PROFILE else '(inherit environment)'}")
show(f"asm profile  {Path(ASSEMBLY_PROFILE).name if ASSEMBLY_PROFILE else '(inherit environment)'}")
show(f"turtle col   {TURTLE_COLUMN or '(auto-detected per source)'}")
show(f"s3 archive   {'set' if S3_ARCHIVE_BUCKET else '(none - artifacts stay under the work dir)'}")
_sector_set = bool(SECTOR_DEFINITIONS_BUCKET and SECTOR_DEFINITIONS_KEY)
show("sector csv   " + ("set" if _sector_set else
      "(UNSET - company bridge, peer edges and sector links will be degraded)"))

## 1. Preflight

Five things account for nearly every adhoc-run failure, and four of them present as a
**hang**, not an error. Check them before spending an hour on a submission.

In [ ]:
def uses_object_storage():
    return any(v.startswith(("s3a://", "s3://"))
               for v in (LOCAL_WORK_DIR, SOURCE_PATHS)) or bool(S3_ARCHIVE_BUCKET)


def preflight():
    ok = True

    def check(label, passed, detail=""):
        nonlocal ok
        ok &= bool(passed)
        show(f"  [{'ok ' if passed else 'FAIL'}] {label}{' - ' + str(detail) if detail else ''}")

    def check_credentials():
        print("credentials")
        # Resolved from each node's own provider chain; the notebook neither reads nor
        # stores a key. Identity is verified but never printed - the caller ARN carries
        # the account id.
        # Run in the repo's venv, not this kernel: that is the interpreter the driver
        # actually uses, and the kernel may not even have boto3 installed.
        probe = ("import boto3;boto3.client('sts').get_caller_identity();"
                 "print('RESOLVED')")
        try:
            out = subprocess.run([str(VENV_PYTHON), "-c", probe],
                                 capture_output=True, text=True, timeout=30)
            resolved = "RESOLVED" in out.stdout
            detail = ("identity confirmed (not printed)" if resolved
                      else out.stderr.strip().splitlines()[-1][:120] if out.stderr.strip()
                      else "no identity returned")
        except Exception as e:
            resolved, detail = False, type(e).__name__
        check("driver credentials resolve", resolved, detail)
        # Executors do NOT inherit the driver's environment: each node resolves S3A
        # credentials through its own chain, so a driver that can read the bucket is no
        # evidence that the other nodes can.
        print("         (executors authenticate per node; the driver's success does not "
              "speak for the rest of the cluster)")

    def check_work_dir(so_far=True):
        if uses_object_storage():
            check_credentials()
        print("output")
        if LOCAL_WORK_DIR.startswith(("s3a://", "s3://")):
            check("work dir is object storage", True, LOCAL_WORK_DIR)
        else:
            # Created here rather than by the job: Spark's committer wants the parent to
            # exist, and a typo'd path is better caught now than 40 minutes in.
            Path(LOCAL_WORK_DIR).mkdir(parents=True, exist_ok=True)
            check("work dir writable", os.access(LOCAL_WORK_DIR, os.W_OK), LOCAL_WORK_DIR)
            if not SPARK_MASTER_URL.startswith("local"):
                print("         (a plain path is only correct if every executor runs on "
                      "this host or it is a shared mount)")
        return so_far and ok

    print("launcher")
    check("bin/submit_spark_job.sh is executable", os.access(LAUNCHER, os.X_OK), str(LAUNCHER))
    check("venv archive built for the executors", VENV_ARCHIVE.is_file(),
          str(VENV_ARCHIVE) if VENV_ARCHIVE.is_file() else "run bin/package_venv.sh - "
          "without it executors use the system python and die with ModuleNotFoundError")
    check("repo venv has torch + torch_geometric", VENV_PYTHON.is_file(), str(VENV_PYTHON))
    # A Jupyter kernel is a non-interactive shell with none of a login shell's profile,
    # so Spark is routinely absent from its PATH. The launcher prefers $SPARK_HOME.
    spark_submit = (Path(SPARK_HOME) / "bin" / "spark-submit") if SPARK_HOME else None
    check("spark-submit reachable",
          bool((spark_submit and spark_submit.is_file()) or shutil.which("spark-submit")),
          str(spark_submit) if spark_submit else (shutil.which("spark-submit")
          or "export SPARK_HOME (e.g. /opt/spark) - the kernel's PATH has no Spark"))

    print("master")
    check("SPARK_MASTER_URL is set", bool(SPARK_MASTER_URL),
          SPARK_MASTER_URL or "export it, e.g. spark://<host>:7077 or local[*]")
    if not SPARK_MASTER_URL:
        return False
    if SPARK_MASTER_URL.startswith("local"):
        print("  [ok ] local master - no cluster checks apply")
        return check_work_dir(ok)

    host, _, port = SPARK_MASTER_URL.replace("spark://", "").partition(":")
    try:
        with socket.create_connection((host, int(port or 7077)), timeout=5):
            check("master accepting connections", True, SPARK_MASTER_URL)
    except OSError as e:
        check("master accepting connections", False, f"{SPARK_MASTER_URL}: {e}")
        return False

    print("workers")
    try:
        with urllib.request.urlopen(f"{MASTER_WEB_UI}/json/", timeout=5) as resp:
            state = json.load(resp)
    except Exception as e:
        print(f"  [warn] could not read {MASTER_WEB_UI}/json/ ({e}); skipping worker checks")
        return check_work_dir()

    alive = [w for w in state.get("workers", []) if w.get("state") == "ALIVE"]
    check("at least one ALIVE worker", alive, f"{len(alive)} alive")

    # A worker that advertises no GPU accepts a GPU-requesting job and then never
    # schedules it. The application sits in WAITING forever with no error.
    with_gpu = [w for w in alive if w.get("resources", {}).get("gpu", {}).get("addresses")]
    all_gpu = len(with_gpu) == len(alive)
    check("workers advertise a GPU", all_gpu, f"{len(with_gpu)}/{len(alive)}" + (
        "" if all_gpu else " - set spark.worker.resource.gpu.amount on the rest"))

    free_cores = sum(w.get("coresfree", 0) for w in alive)
    check("cores free", free_cores > 0,
          f"{free_cores} free / {sum(w.get('cores', 0) for w in alive)} total")

    # Standalone Spark hands an application every core by default, so one orphaned
    # driver from an earlier killed run starves everything after it.
    running = [a for a in state.get("activeapps", []) if a.get("state") == "RUNNING"]
    check("no application already holding the cluster", not running,
          ", ".join(f"{a['name']} ({a.get('cores')} cores)" for a in running) or "idle")

    return check_work_dir()


PREFLIGHT_OK = preflight()
print("\nPREFLIGHT", "OK" if PREFLIGHT_OK else "FAILED - fix the above before submitting")

## 2. The submit helper

One function, used by both the seed run and every experiment. It streams output as it
arrives (a 40-minute submission with no output is indistinguishable from a hang), keeps
the full log for later inspection, and kills the whole process group on timeout.

In [ ]:
RUNS = []   # every submission this session, in order

# The driver logs one of these per executor as it registers. Parsing them is the only
# way to find out, after the fact, how many machines actually did the work.
_EXECUTOR_HOST_RES = [
    re.compile(r"Registered executor \S+ \(([^:)\s]+):\d+\) with ID"),
    re.compile(r"Executor added: \S+ on ([^:\s]+):\d+"),
    re.compile(r"Launching executor \S+ on worker \S+ \(([^:)\s]+):\d+\)"),
]


def executor_hosts(output):
    """Distinct hosts that ran an executor for this submission.

    A cluster submission that reports ONE host did not fan out, whatever the master
    UI showed: in client mode the driver advertises an address it guesses, and an
    executor that cannot dial back is killed and relaunched forever while the one
    that can runs the entire job. The job succeeds. Set SPARK_DRIVER_HOST to the
    driver's address on the master's network and resubmit.
    """
    hosts = set()
    for pattern in _EXECUTOR_HOST_RES:
        hosts.update(pattern.findall(output))
    return sorted(hosts)


def explain_is_on():
    """True when RAPIDS is emitting its per-operator GPU/CPU decisions.

    Those lines are the only place the REASON for a fallback appears, and they
    exist only at spark.rapids.sql.explain=ALL.
    """
    return RAPIDS_EXPLAIN.strip().upper() == "ALL"


# Standalone Spark logs this line once per submission. It is the only link
# between a run and the event log file the cluster wrote for it.
_APP_ID_RE = re.compile(r"app ID (app-[\w-]+)")
_NODE_NAME_RE = re.compile(r'"nodeName":"(\w+)"')


def event_log_dir():
    """The local directory Spark writes event logs to, or None.

    Read from SPARK_EXTRA_CONF first -- that is where a run script turns event
    logging on -- and from spark-defaults.conf second. A remote destination
    reads as None: this opens the files directly and does not fetch.
    """
    sources = [os.environ.get("SPARK_EXTRA_CONF", "")]
    defaults = Path(SPARK_HOME or "/opt/spark") / "conf" / "spark-defaults.conf"
    if defaults.is_file():
        sources.append(defaults.read_text(errors="replace"))

    for text in sources:
        match = re.search(r"spark\.eventLog\.dir[=\s]+(\S+)", text)
        if not match:
            continue
        path = match.group(1).strip("\"'")
        if path.startswith("file://"):
            path = path[len("file://"):]
        return None if "://" in path else Path(path)
    return None


def gpu_operator_counts(record):
    """How many plan operators the RAPIDS plugin took, from the event log.

    The per-operator explain log answers this too, at 306 MB / 1.9M lines. The
    event log carries the same answer in the plans themselves: every operator
    the plugin claimed is named Gpu*. Counts operator INSTANCES over every plan
    in the run, so the pair is a ratio and not a census of distinct operators.

    Returns {"gpu": int, "other": int}, or None when the log is unreachable --
    event logging off, written somewhere remote, or no decompressor for it.
    """
    directory = event_log_dir()
    app_id = _APP_ID_RE.search(record.get("output", "") or "")
    if directory is None or app_id is None or not directory.is_dir():
        return None

    files = sorted(directory.glob(f"{app_id.group(1)}*"))
    if not files:
        return None

    path, proc, stream = files[0], None, None
    # Spark compresses these when spark.eventLog.compress is on, and the repo
    # venv has no codec for either format -- hand it to the CLI tool instead.
    decoder = {".zstd": "zstdcat", ".zst": "zstdcat", ".lz4": "lz4"}.get(
        path.suffix
    )
    try:
        if decoder:
            args = [decoder, str(path)] if decoder == "zstdcat" else [
                decoder, "-dc", str(path)]
            proc = subprocess.Popen(args, stdout=subprocess.PIPE, text=True,
                                    stderr=subprocess.DEVNULL)
            stream = proc.stdout
        else:
            stream = path.open(errors="replace")

        gpu = other = 0
        for line in stream:
            for name in _NODE_NAME_RE.findall(line):
                if name.startswith("Gpu"):
                    gpu += 1
                else:
                    other += 1
    except Exception:
        return None
    finally:
        if stream is not None:
            stream.close()
        if proc is not None:
            proc.wait()

    return {"gpu": gpu, "other": other}


def gpu_summary(record):
    """One phrase for where a submission's operators actually ran.

    Says "not measured" rather than "no GPU" when there was nothing to count.
    Zero matched lines is the absence of a measurement, and the two point at
    opposite next steps: one is a plugin that never loaded, the other is a log
    level. The event log is consulted first, because it answers either way.
    """
    if record.get("gpu_ops") is not None:
        return (f"{record['gpu_ops']} ops on GPU, "
                f"{record['cpu_fallbacks']} CPU fallbacks")

    counts = gpu_operator_counts(record)
    if counts is not None:
        total = counts["gpu"] + counts["other"]
        share = (100 * counts["gpu"] / total) if total else 0
        return (f"{counts['gpu']:,} of {total:,} plan operators on GPU "
                f"({share:.0f}%, counted in the event log)")

    return ("GPU placement NOT MEASURED - no per-operator lines "
            f"(spark.rapids.sql.explain={record.get('rapids_explain', '?')}) "
            "and no readable event log")


def profile_env(path):
    """The variables a sizing profile exports, for one submission's environment.

    Sourced in a subshell rather than parsed, so a profile is free to compute a value.
    Only what it actually CHANGES is returned, so a sizing file can never quietly
    redefine an identity variable -- master URL, driver host, bucket paths -- that the
    caller set.
    """
    if not path:
        return {}
    dump = "import json,os,sys; json.dump(dict(os.environ), sys.stdout)"
    out = subprocess.run(
        ["bash", "-c", 'set -a; . "$1"; set +a; exec "$2" -c "$3"',
         "_", str(path), sys.executable, dump],
        capture_output=True, text=True, check=True,
    ).stdout
    return {k: v for k, v in json.loads(out).items() if os.environ.get(k) != v}


def submit(*, mode, pyg_filename=None, pyg_config=None, source_paths=None,
           timeout_s=EXPERIMENT_TIMEOUT_S, label=None, extra_args=(), dry_run=False,
           profile=None):
    """Submit one job through bin/submit_spark_job.sh and return a record of the run.

    Returns a dict: label, cmd, returncode, seconds, output, gpu_ops, cpu_fallbacks,
    pyg_path. Nothing raises on job failure - inspect the record.
    """
    label = label or f"{mode}:{pyg_filename or 'default'}"

    cmd = [str(LAUNCHER), "--mode", mode,
           "--local_work_dir", LOCAL_WORK_DIR,
           "--time_period", TIME_PERIOD]

    if mode in ("full", "enrichment_only"):
        paths = source_paths if source_paths is not None else SOURCE_PATHS
        if not paths:
            raise ValueError(f"--mode {mode} needs source_paths (set SOURCE_PATHS in the config cell)")
        cmd += ["--source_paths", paths,
                "--source_format", SOURCE_FORMAT,
                "--parquet_partitions", str(PARQUET_PARTITIONS)]
        # Passed only when forced. Omitted, the job resolves the column per source, which
        # is what lets one run mix sources that named the Turtle blob differently.
        if SOURCE_FORMAT == "turtle_parquet" and TURTLE_COLUMN:
            cmd += ["--turtle_column", TURTLE_COLUMN]
        # Only the enrichment phase reads these, so they are passed on the seed leg
        # alone. Omitted, the job runs and produces a less connected graph in silence.
        if SECTOR_DEFINITIONS_BUCKET and SECTOR_DEFINITIONS_KEY:
            cmd += ["--market_sector_definitions_bucket", SECTOR_DEFINITIONS_BUCKET,
                    "--market_sector_definitions_key", SECTOR_DEFINITIONS_KEY]

    if pyg_filename:
        cmd += ["--pyg_filename", pyg_filename]
    if pyg_config:
        cmd += ["--pyg_config", json.dumps(pyg_config, separators=(",", ":"))]
    if S3_ARCHIVE_BUCKET and mode in ("full", "pyg_only"):
        cmd += ["--s3_archive_bucket", S3_ARCHIVE_BUCKET]
    cmd += list(extra_args)

    overlay = profile_env(profile)
    print(f"=== {label}")
    show("    " + " ".join(cmd))
    if profile:
        # Recorded per submission because the two legs use different sizing, and a
        # run that OOMs is unreadable without knowing which one it had.
        print(f"    profile {Path(profile).name}: "
              + " ".join(f"{k}={v}" for k, v in sorted(overlay.items())
                         if k.startswith(("EXECUTOR_", "DRIVER_", "RAPIDS_", "GPU_",
                                          "PYSPARK_ARROW"))))
    if dry_run:
        return {"label": label, "cmd": cmd, "dry_run": True}

    env = {**os.environ, **overlay,
           "SPARK_MASTER_URL": SPARK_MASTER_URL,
           "RAPIDS_EXPLAIN": RAPIDS_EXPLAIN}
    # Left unset rather than guessed: the launcher only passes --conf spark.driver.host
    # when this is non-empty, and Spark's own guess is right on a single-NIC host.
    if SPARK_DRIVER_HOST:
        env["SPARK_DRIVER_HOST"] = SPARK_DRIVER_HOST
    if SPARK_MASTER_URL.startswith("local"):
        # local[*] skips the venv archive (there are no remote executors to ship it to),
        # so spark-submit would run whatever python it finds -- usually one without torch
        # or PyG, failing at import. Point both ends at the repo's venv.
        env["PYSPARK_PYTHON"] = env["PYSPARK_DRIVER_PYTHON"] = str(VENV_PYTHON)

    # start_new_session: the launcher and the spark-submit JVM it spawns share a process
    # group we can kill as a unit. Kill only the shell and the driver survives, holding
    # every core on the cluster - the next submission then looks like a hang.
    started = time.monotonic()
    proc = subprocess.Popen(
        cmd, cwd=REPO_ROOT, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True,
    )

    lines, timed_out = [], False
    try:
        deadline = started + timeout_s
        for line in proc.stdout:
            lines.append(line)
            if STREAM_TAIL_LINES == 0 or any(f in line for f in STREAM_FILTER):
                # Spark logs its own paths and peer addresses; redact on the way out.
                show("   ", line.rstrip())
            if time.monotonic() > deadline:
                timed_out = True
                break
        if timed_out:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        proc.wait(timeout=60)
    except KeyboardInterrupt:
        os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        proc.wait(timeout=60)
        print("    INTERRUPTED - driver killed with its process group")
        timed_out = True

    output = "".join(lines)
    elapsed = time.monotonic() - started

    record = {
        "label": label,
        "mode": mode,
        "cmd": cmd,
        "profile": Path(profile).name if profile else None,
        "pyg_filename": pyg_filename,
        "pyg_config": pyg_config,
        "returncode": None if timed_out else proc.returncode,
        "timed_out": timed_out,
        "seconds": round(elapsed, 1),
        "output": output,
        # RAPIDS' own per-operator decisions. It emits these only when
        # spark.rapids.sql.explain is ALL; at any other level there is nothing to
        # count, and zero matches means NOT MEASURED. None says exactly that, so
        # no reader downstream can turn an absent log line into "ran on the CPU".
        "rapids_explain": RAPIDS_EXPLAIN,
        "gpu_ops": output.count("will run on GPU") if explain_is_on() else None,
        "cpu_fallbacks": (output.count("cannot run on GPU")
                          if explain_is_on() else None),
        "executor_hosts": executor_hosts(output),
        "pyg_path": pyg_output_path(pyg_filename) if mode in ("full", "pyg_only") else None,
    }
    RUNS.append(record)

    if timed_out:
        print(f"    HUNG after {timeout_s}s - killed. Usual causes: workers advertise no "
              f"GPU while every task requests one, or an orphaned driver holds the cores.")
    else:
        status = "ok" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
        hosts = record["executor_hosts"]
        print(f"    {status} in {elapsed / 60:.1f} min | "
              f"{gpu_summary(record)} | "
              f"executors on {len(hosts)} host(s)")
        if not SPARK_MASTER_URL.startswith("local") and len(hosts) < 2:
            print("    WARNING: fewer than two hosts ran an executor. If the cluster has "
                  "more than one worker, this run did NOT fan out - see the fan-out "
                  "section below.")
        if proc.returncode != 0:
            print("    --- tail ---")
            show("\n".join("    " + l for l in output.splitlines()[-25:]))

    return record


def period_partition(period=None):
    """Mirror of build_graph.period_partition: '2026-07' -> 'year=2026/month=07'."""
    period = period or TIME_PERIOD
    parts = period.split("-")
    if len(parts) == 2 and len(parts[0]) == 4 and len(parts[1]) == 2:
        return f"year={parts[0]}/month={parts[1]}"
    return period


def pyg_output_path(pyg_filename=None):
    """Where the job writes the .pt, derived exactly as JobConfig does."""
    return (f"{LOCAL_WORK_DIR.rstrip('/')}/pyg/{period_partition()}/"
            f"{pyg_filename or 'hetero_data.pt'}")


def enriched_parquet_path():
    return f"{LOCAL_WORK_DIR.rstrip('/')}/enriched/{period_partition()}/triples"

## 3. Seed run — produce the enriched Parquet

Run this **once** per period. Everything in section 4 reads the Parquet it writes, so a
seed at `--mode enrichment_only` is enough to unlock the experiment loop; `--mode full`
additionally produces a baseline `hetero_data.pt` and exercises the driver-side artifact
writers (`.pt`, six metadata JSONs, node index, manifest) against real shared storage.

Skip it if `enriched_parquet_path()` already has data for this period **and that data
was written by the current code**. Enrichment is where the cross-source structure is
built, so Parquet from before an enrichment change does not contain what that change
added, and a `pyg_only` loop over it will show none of it. The PyG builder is read
from the repo on every submission, so only the *enrichment* half goes stale this way.
When in doubt, re-seed: it is one run, and every experiment after it inherits the
answer.

In [ ]:
# enrichment_only, not full. --mode full tacks a PyG assembly onto the end of the seed,
# and assembly needs the driver-heavy profile while the enrichment ahead of it needs the
# executor-heavy one -- a single submission cannot have both, and the pairing that loses
# is assembly. That is exactly how the 2026-08-25 seed spent 182 minutes enriching
# correctly and then died building the graph.
#
# Nothing is given up: the driver-side artifact writers (.pt, six metadata JSONs, node
# index, manifest) all run for real in every experiment in section 4.
SEED_MODE = "enrichment_only"

seed = submit(
    mode=SEED_MODE,
    label=f"seed ({SEED_MODE}, {TIME_PERIOD})",
    timeout_s=SEED_TIMEOUT_S,
    profile=SEED_PROFILE,
    dry_run=not PREFLIGHT_OK,   # refuse to burn an hour on a cluster that failed preflight
)

## 4. Experiment matrix

Each entry is one `--mode pyg_only` submission over the **same** enriched Parquet, so the
only variable is the PyG config. Distinct `--pyg_filename` per experiment keeps the
outputs side by side (`hetero_data_512d.pt`, its metadata in
`hetero_data_512d_metadata/`) instead of overwriting one another.

Add, remove, or reorder freely — this is the part meant to be edited.

In [ ]:
# ORDERED CHEAPEST FIRST, deliberately. Each of these holds its whole graph in the
# driver's memory, and they do not cost the same: on the 2026-08 graph (10,348,460 nodes,
# 76,764,031 edges) the node features are 42.4 GiB in every leg, and the edge features
# add ~9.8 GiB on top for the legs that build them. no_edge_features is therefore the
# probe -- if the cheap one cannot fit, the expensive ones certainly cannot, and finding
# that out first costs 12 minutes instead of an hour.
#
# A third leg, all_edge_categories, was dropped on 2026-08-29. It added "causal" to the
# default four and spent 80 minutes rebuilding what baseline_1024d had already built:
# nothing in this ontology classifies as causal. leadsTo, impacts and causes match no
# relation at all, and both "affects" relations -- bls_enrichment_affectsRegion and
# noaa_enrichment_affectsSameRegion -- are on _SKIP_RELATION_FRAGMENTS, which
# _classify_relation checks before any category. Same 9 edge types either way.
#
# "generic" is the one category left that would change this graph, and it is not a
# drop-in: 264 edge types and 56.4M edges against the 9 and 19.7M the default four
# cover. Size it on its own before giving it a leg here.
EXPERIMENTS = [
    {
        "name": "no_edge_features",
        "pyg_filename": "hetero_data_no_edge_features.pt",
        "pyg_config": {"edge_feature_config": {"enabled": False}},
    },
    {
        # Every value here is already the default -- vector_dim 1024, normalize True,
        # edge_vector_dim 32. Restated so the printed config records what was built
        # instead of sending a reader to look the defaults up.
        "name": "baseline_1024d",
        "pyg_filename": "hetero_data_1024d.pt",
        "pyg_config": {
            "feature_config": {"vector_dim": 1024, "normalize": True},
            "edge_feature_config": {"edge_vector_dim": 32},
        },
    },
]

for e in EXPERIMENTS:
    show(f"{e['name']:24} -> {pyg_output_path(e['pyg_filename'])}")
    print(f"{'':24}    {json.dumps(e['pyg_config'])}")

In [ ]:
# Submit them one at a time. Standalone Spark gives an application every core by default,
# so concurrent submissions queue rather than parallelize - serial is both simpler and
# what actually happens.
results = {}
for experiment in EXPERIMENTS:
    results[experiment["name"]] = submit(
        mode="pyg_only",
        label=experiment["name"],
        pyg_filename=experiment["pyg_filename"],
        pyg_config=experiment["pyg_config"],
        timeout_s=EXPERIMENT_TIMEOUT_S,
        profile=ASSEMBLY_PROFILE,
        dry_run=not PREFLIGHT_OK,
    )
    print()

print(f"{'experiment':26} {'status':10} {'minutes':>8} {'gpu ops':>8} {'cpu fb':>7}")
for name, r in results.items():
    if r.get("dry_run"):
        status = "dry-run"
    elif r["timed_out"]:
        status = "HUNG"
    else:
        status = "ok" if r["returncode"] == 0 else f"exit {r['returncode']}"
    # "-" is not zero. With explain off there were no per-operator lines to
    # count, and printing 0 there reads as a finding the run never made.
    gpu_ops, cpu_fb = r.get("gpu_ops"), r.get("cpu_fallbacks")
    print(f"{name:26} {status:10} {r.get('seconds', 0) / 60:8.1f} "
          f"{'-' if gpu_ops is None else gpu_ops:>8} "
          f"{'-' if cpu_fb is None else cpu_fb:>7}")

if any(r.get("gpu_ops") is None and not r.get("dry_run") for r in results.values()):
    print(f"\n(gpu columns read '-' because spark.rapids.sql.explain={RAPIDS_EXPLAIN} "
          f"emitted no per-operator lines -- not measured, not zero. Section 8 counts "
          f"the plan operators in the event log instead.)")

## 5. Read the artifacts back

Exit code 0 is not evidence of a usable graph. This section opens what the cluster
actually wrote.

The `.pt` unpickles into a `torch_geometric.data.HeteroData`, so it needs
`torch-geometric` — which the JupyterLab kernel does not have. The inspection therefore
runs in the **repo's own venv** (`.venv/bin/python`) as a subprocess and returns JSON,
rather than requiring you to install PyG here. Same interpreter the job's driver uses, so
what loads there loads here.

In [ ]:
INSPECT_SRC = r"""
import io, json, os, sys

path = sys.argv[1]
summary = {"path": path}

def read_bytes(uri):
    if uri.startswith(("s3a://", "s3://")):
        import boto3
        bucket, _, key = uri.split("://", 1)[1].partition("/")
        buf = io.BytesIO()
        boto3.client("s3").download_fileobj(bucket, key, buf)
        return buf.getvalue()
    with open(uri, "rb") as fh:
        return fh.read()

def list_metadata(pt_uri):
    stem = os.path.basename(pt_uri)[: -len(".pt")]
    base = pt_uri.rsplit("/", 1)[0]
    # build_graph writes 'metadata/' for the default filename, '<stem>_metadata/' otherwise
    for candidate in (f"{base}/{stem}_metadata", f"{base}/metadata"):
        try:
            names = {}
            for fname in ("graph_schema.json", "feature_spec.json", "normalization.json",
                          "encoding_config.json", "ontology_schema.json", "slot_mapping.json"):
                names[fname] = json.loads(read_bytes(f"{candidate}/{fname}"))
            return candidate, names
        except Exception:
            continue
    return None, {}

try:
    import torch
    raw = read_bytes(path)
    summary["bytes"] = len(raw)
    data = torch.load(io.BytesIO(raw), weights_only=False)

    summary["node_types"] = {t: int(data[t].num_nodes) for t in data.node_types}
    summary["node_feature_dims"] = {
        t: list(data[t].x.shape) for t in data.node_types if "x" in data[t]
    }
    edges = {}
    for et in data.edge_types:
        store = data[et]
        entry = {"num_edges": int(store.edge_index.shape[1])}
        if "edge_attr" in store:
            entry["edge_attr_dim"] = int(store.edge_attr.shape[1])
        # A dangling index is the failure that a file-exists check cannot see.
        src, dst = et[0], et[2]
        entry["index_in_range"] = bool(
            store.edge_index.numel() == 0 or (
                int(store.edge_index[0].max()) < data[src].num_nodes
                and int(store.edge_index[1].max()) < data[dst].num_nodes
                and int(store.edge_index.min()) >= 0
            )
        )
        edges["__".join(et)] = entry
    summary["edge_types"] = edges
    summary["total_nodes"] = sum(summary["node_types"].values())
    summary["total_edges"] = sum(e["num_edges"] for e in edges.values())
    summary["all_features_finite"] = bool(all(
        torch.isfinite(data[t].x).all() for t in data.node_types if "x" in data[t]
    ))
    summary["float32_features"] = bool(all(
        data[t].x.dtype == torch.float32 for t in data.node_types if "x" in data[t]
    ))
    summary["dangling_edge_types"] = [k for k, v in edges.items() if not v["index_in_range"]]

    md_dir, metadata = list_metadata(path)
    summary["metadata_dir"] = md_dir
    summary["metadata_files"] = sorted(metadata)
    # Keyed by FILENAME, extension included -- list_metadata() above stores it that
    # way, and asking for "feature_spec" silently returns {} for every graph.
    spec = metadata.get("feature_spec.json", {})
    # feature_spec.json states the width under <section>.total_dim; there is no
    # top-level "vector_dim" key, so reading one reports None for every graph.
    summary["feature_spec_node_dim"] = spec.get("node_features", {}).get("total_dim")
    summary["feature_spec_edge_dim"] = spec.get("edge_features", {}).get("total_dim")
except Exception as exc:
    summary["error"] = f"{type(exc).__name__}: {exc}"

print(json.dumps(summary))
"""


def inspect_pt(pt_path):
    """Load a .pt (+ its metadata) in the repo venv; return a JSON-able summary."""
    proc = subprocess.run(
        [str(VENV_PYTHON), "-c", INSPECT_SRC, pt_path],
        cwd=REPO_ROOT, capture_output=True, text=True,
    )
    if proc.returncode != 0:
        return {"path": pt_path, "error": proc.stderr.strip()[-800:]}
    return json.loads(proc.stdout.splitlines()[-1])


summaries = {}
for name, r in results.items():
    if r.get("dry_run") or r.get("returncode") != 0:
        print(f"(skip {name}: did not complete)")
        continue
    summaries[name] = inspect_pt(r["pyg_path"])
    scalars = {k: v for k, v in summaries[name].items() if not isinstance(v, (dict, list))}
    show(f"{name}: {json.dumps(scalars)}")

In [ ]:
# Side-by-side comparison of the graphs the cluster just built.
def cell(v):
    return "-" if v is None else str(v)

rows = [
    ("bytes", lambda s: f"{s.get('bytes', 0):,}"),
    ("node types", lambda s: len(s.get("node_types", {}))),
    ("nodes", lambda s: f"{s.get('total_nodes', 0):,}"),
    ("edge types", lambda s: len(s.get("edge_types", {}))),
    ("edges", lambda s: f"{s.get('total_edges', 0):,}"),
    ("node feat dim", lambda s: sorted({d[1] for d in s.get("node_feature_dims", {}).values()})),
    ("edge feat dims", lambda s: sorted({e["edge_attr_dim"] for e in s.get("edge_types", {}).values()
                                         if "edge_attr_dim" in e})),
    ("edge types w/ feats", lambda s: sum(1 for e in s.get("edge_types", {}).values()
                                          if "edge_attr_dim" in e)),
    ("features finite", lambda s: s.get("all_features_finite")),
    ("float32", lambda s: s.get("float32_features")),
    ("dangling edges", lambda s: s.get("dangling_edge_types") or "none"),
    ("metadata files", lambda s: len(s.get("metadata_files", []))),
]

names = list(summaries)
width = max([12] + [len(n) for n in names])
print(f"{'':22} " + " ".join(f"{n:>{width}}" for n in names))
for label, fn in rows:
    print(f"{label:22} " + " ".join(f"{cell(fn(summaries[n])):>{width}}" for n in names))

missing = {n: set(("graph_schema.json", "feature_spec.json", "normalization.json",
                   "encoding_config.json", "ontology_schema.json", "slot_mapping.json"))
              - set(summaries[n].get("metadata_files", []))
           for n in names}
for n, m in missing.items():
    if m:
        print(f"\n{n}: MISSING metadata {sorted(m)}")

## 6. Per-experiment detail

Drill into one graph — node/edge inventory as the cluster built it.

In [ ]:
FOCUS = next(iter(summaries), None)

if FOCUS:
    s = summaries[FOCUS]
    show(f"{FOCUS}  ->  {s['path']}\n")
    print("node types")
    for t, n in sorted(s["node_types"].items(), key=lambda kv: -kv[1]):
        dim = s.get("node_feature_dims", {}).get(t)
        print(f"  {t:40} {n:>10,}  x={dim}")
    print("\nedge types")
    for et, e in sorted(s["edge_types"].items(), key=lambda kv: -kv[1]["num_edges"]):
        feat = f"  edge_attr={e['edge_attr_dim']}d" if "edge_attr_dim" in e else ""
        print(f"  {et:60} {e['num_edges']:>10,}{feat}")
else:
    print("no completed experiment to inspect")

## 7. Did it actually fan out?

The master's worker list showing every worker `ALIVE` is not evidence that every worker
*ran* anything. In client mode the driver advertises an address it picks from its own
interfaces; executors that cannot reach it time out after 120s, exit 1, and are relaunched
forever, while the executor co-located with the driver reaches it over loopback and runs
the entire job. The application succeeds, the artifacts are correct, and the cluster ran
at the capacity of one machine.

The driver logs each executor's address as it registers, so counting distinct hosts is the
direct check. Fewer hosts than workers means set `SPARK_LOCAL_IP` per node (and
`SPARK_DRIVER_HOST` here) to the addresses on the network the cluster actually uses, then
resubmit. Addresses are redacted below — only the count matters.

In [ ]:
expected_workers = int(os.environ.get("PYG_EXPECTED_WORKERS", "0"))  # 0 = don't assert

for r in RUNS:
    if r.get("dry_run"):
        continue
    hosts = r.get("executor_hosts", [])
    verdict = f"{len(hosts)} host(s)"
    if SPARK_MASTER_URL.startswith("local"):
        verdict += "  (local master - single process by definition)"
    elif len(hosts) < 2:
        verdict += "  <- did NOT fan out; every task ran on one machine"
    elif expected_workers and len(hosts) < expected_workers:
        verdict += f"  <- fewer than the {expected_workers} workers expected"
    show(f"{r['label']:30} {verdict}")

# Nothing here prints an address. If you need them while debugging, they are in
# RUNS[-1]["executor_hosts"] - just don't commit that cell's output.
if RUNS and not RUNS[-1].get("dry_run") and not RUNS[-1].get("executor_hosts"):
    print("\nNo executor registration lines were found in the driver log. Either the run "
          "never got as far as scheduling (check preflight), or the driver's log level "
          "hides them - INFO on org.apache.spark.scheduler is what emits them.")

## 8. GPU placement

A run that fell back to the CPU produces the *same graph* and exits 0, so placement has to
be measured rather than inferred.

Two sources, in order. RAPIDS' own per-operator log (`RAPIDS_EXPLAIN=ALL`) is the only one
that gives the *reason* for a fallback, and it costs 306 MB / 1.9M lines on a full run.
The Spark **event log** answers *where* a run executed at any explain level, because every
operator the plugin claimed is named `Gpu*` in the plan.

When neither is available this section says **not measured** — never "no GPU". Zero
matched log lines is an absent measurement, and the two point at opposite next steps: a
plugin that never loaded (missing or mismatched jar, a worker advertising no GPU) versus a
log level that was turned down on purpose.

In [ ]:
for r in RUNS:
    if r.get("dry_run"):
        continue
    print(f"{r['label']:30} {gpu_summary(r)}")

# What fell back, and why - the reason RAPIDS gives is usually the fix. Only ALL
# emits it; with the level turned down there is nothing to list, and saying so beats
# printing "0 CPU fallbacks" over a log that was never asked to record any.
LAST = RUNS[-1] if RUNS else None
if LAST and not LAST.get("dry_run"):
    if LAST.get("cpu_fallbacks") is None:
        print(f"\n{LAST['label']}: per-operator fallback reasons need "
              f"RAPIDS_EXPLAIN=ALL; this run set "
              f"{LAST.get('rapids_explain', '?')}.")
    else:
        reasons = [l.strip() for l in LAST["output"].splitlines()
                   if "cannot run on GPU" in l]
        print(f"\n{LAST['label']}: {len(reasons)} CPU fallbacks; first few:")
        for line in reasons[:15]:
            show("  " + line[:200])

## 9. Scratch

Anything adhoc: re-run a single experiment, submit a one-off config, or dump a full log.

```python
# one-off submission
submit(mode="pyg_only", label="scratch", pyg_filename="hetero_data_scratch.pt",
       pyg_config={"feature_config": {"vector_dim": 256}})

# full log of the last run - UNREDACTED, contains paths and peer addresses.
# Clear this cell's output before committing.
print(RUNS[-1]["output"])

# inspect a .pt this notebook did not build
inspect_pt(pyg_output_path("hetero_data.pt"))
```

**Before committing this notebook**, clear its outputs unless you have read them:

```bash
jupyter nbconvert --clear-output --inplace notebook/multi_experiment.ipynb
```

Everything the cells print is redacted, but a stray `print()` you added while debugging is
not, and outputs are stored in the `.ipynb`.

**The `latest` alias.** Every `full` / `pyg_only` submission also writes
`graph_schema.json` to `pyg/latest/<stem>_metadata/` - a fixed key that always names
the most recent build, so a consumer can fetch one URL without knowing which period
is newest. Only the schema is aliased, and each `--pyg_filename` gets its own alias,
so experiment variants never collide. It is written to the work dir; the optional S3
archive mirror does not carry it.

**When you're done:** the enriched Parquet under `enriched_parquet_path()` is the
expensive artifact — keep it and the experiment loop stays cheap. The per-experiment
`.pt` files are disposable; delete the ones you're not comparing.

In [ ]:
# Housekeeping: what this session produced, and where.
show(f"enriched parquet   {enriched_parquet_path()}")
for r in RUNS:
    if r.get("pyg_path") and not r.get("dry_run") and r.get("returncode") == 0:
        show(f"{r['label']:30} {r['pyg_path']}")